# Check sub-SOCO ⊂ SOCO
Strategia: (1) match esatto via SHA1 dei byte (se i file sono solo rinominati), (2) fallback con pHash + stessa dimensione per i residui, (3) differenza insiemistica per trovare le immagini SOCO non presenti nel subset.

In [1]:
# !pip install pillow imagehash tqdm numpy pandas
from pathlib import Path
import hashlib, pickle
import numpy as np, pandas as pd
from PIL import Image
import imagehash
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor

ROOT = Path("datasets")          # adatta se il notebook sta altrove
SOCO = ROOT / "SOCO"
SUB  = ROOT / "sub-SOCO"
CACHE = ROOT / "hash_cache.pkl"
HASH_SIZE = 16                   # 256 bit
PHASH_THR = 10                   # soglia Hamming (su 256 bit), da tarare guardando l'istogramma
EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

In [2]:
def list_images(base):
    return sorted(p for p in tqdm(base.rglob("*"), desc=f"scan {base.name}") if p.suffix in EXTS)

soco_files = list_images(SOCO / "Images")
sub_files  = list_images(SUB / "images")
print(len(soco_files), "immagini SOCO;", len(sub_files), "immagini sub-SOCO")
print("esempi SOCO:", [str(p.relative_to(SOCO)) for p in soco_files[:3]])
print("esempi sub :", [str(p.relative_to(SUB)) for p in sub_files[:3]])

# struttura annotazioni (utile per ricostruire coppie/split originali)
for d in sorted((SOCO / "PairAnnotations").iterdir()):
    items = sorted(d.iterdir())[:3] if d.is_dir() else [d]
    print(d.name, "->", [i.name for i in items])

scan Images: 0it [00:00, ?it/s]

scan images: 0it [00:00, ?it/s]

4000 immagini SOCO; 646 immagini sub-SOCO
esempi SOCO: ['Images/aeroplane/aeroplane_1.JPEG', 'Images/aeroplane/aeroplane_10.JPEG', 'Images/aeroplane/aeroplane_11.JPEG']
esempi sub : ['images/test/img_0000.jpeg', 'images/test/img_0002.jpeg', 'images/test/img_0009.jpeg']
cross -> ['aeroplane', 'ambulance', 'american_black_bear']
intra -> ['aeroplane', 'ambulance', 'american_black_bear']
trainsplits -> ['test', 'train']


In [3]:
def describe(path):
    path = Path(path)
    data = path.read_bytes()
    sha = hashlib.sha1(data).hexdigest()
    with Image.open(path) as im:
        size = im.size                      # (w, h) originali
        im = im.convert("RGB")
        ph = imagehash.phash(im, hash_size=HASH_SIZE)
    bits = np.packbits(ph.hash.flatten())   # 32 byte
    return str(path), sha, size, bits

def describe_all(files, desc):
    with ProcessPoolExecutor() as ex:
        out = list(tqdm(ex.map(describe, files, chunksize=16), total=len(files), desc=desc))
    return pd.DataFrame(out, columns=["path", "sha1", "size", "bits"])

if CACHE.exists():
    df_soco, df_sub = pickle.load(open(CACHE, "rb"))
else:
    df_soco = describe_all(soco_files, "hash SOCO")
    df_sub  = describe_all(sub_files,  "hash sub-SOCO")
    pickle.dump((df_soco, df_sub), open(CACHE, "wb"))
df_sub["split"] = df_sub.path.str.contains("/train/").map({True: "train", False: "test"})

hash SOCO:   0%|          | 0/4000 [00:00<?, ?it/s]

hash sub-SOCO:   0%|          | 0/646 [00:00<?, ?it/s]

In [4]:
# 1) match esatto sui byte
sha2idx = df_soco.groupby("sha1").indices
exact = {}
for i, s in tqdm(enumerate(df_sub.sha1), total=len(df_sub), desc="sha1 match"):
    if s in sha2idx:
        exact[i] = int(sha2idx[s][0])
print(f"match esatti: {len(exact)}/{len(df_sub)}")

sha1 match:   0%|          | 0/646 [00:00<?, ?it/s]

match esatti: 646/646


In [5]:
# 2) fallback pHash per i residui (stessa dimensione richiesta)
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
soco_bits = np.stack(df_soco.bits.values)            # (M, 32)
soco_size = np.array(df_soco["size"].tolist())       # (M, 2)

phash_match, rows = {}, []
for i in tqdm([i for i in range(len(df_sub)) if i not in exact], desc="phash match"):
    b = df_sub.bits.iloc[i]; sz = np.array(df_sub["size"].iloc[i])
    d = POP[np.bitwise_xor(soco_bits, b)].sum(1).astype(int)
    d[(soco_size != sz).any(1)] = 10_000             # dimensioni diverse -> escluso
    order = np.argsort(d)[:2]
    best, second = order[0], order[1]
    rows.append((i, int(best), d[best], d[second]))
    if d[best] <= PHASH_THR:
        phash_match[i] = int(best)

df_ph = pd.DataFrame(rows, columns=["sub_idx", "soco_idx", "dist", "dist_second"])
print(f"match pHash: {len(phash_match)}/{len(rows)} residui")
if len(df_ph):
    print(df_ph.dist.describe())
    print("ambigui (best e second vicini):", (df_ph.dist_second - df_ph.dist < 4).sum())

phash match: 0it [00:00, ?it/s]

match pHash: 0/0 residui


In [6]:
# 3) risultato finale
mapping = {**exact, **phash_match}
unmatched = [i for i in range(len(df_sub)) if i not in mapping]
print(f"mappate {len(mapping)}/{len(df_sub)}; non mappate: {len(unmatched)}")
print("collisioni (due immagini sub -> stessa SOCO):", len(mapping) - len(set(mapping.values())))

df_map = pd.DataFrame({
    "sub_path":  [df_sub.path[i] for i in mapping],
    "split":     [df_sub.split[i] for i in mapping],
    "soco_path": [df_soco.path[j] for j in mapping.values()],
    "how":       ["sha1" if i in exact else "phash" for i in mapping],
})
df_map.to_csv(ROOT / "sub_to_soco_mapping.csv", index=False)

used = set(mapping.values())
df_missing = df_soco[~df_soco.index.isin(used)][["path", "size"]]
df_missing.to_csv(ROOT / "soco_images_not_in_sub.csv", index=False)
print("immagini SOCO NON presenti in sub-SOCO:", len(df_missing))
display(df_map.how.value_counts())

mappate 646/646; non mappate: 0
collisioni (due immagini sub -> stessa SOCO): 0
immagini SOCO NON presenti in sub-SOCO: 3354


how
sha1    646
Name: count, dtype: int64

In [7]:
# 4) sanity check: da quali cartelle/categorie SOCO vengono train e test?
df_map["soco_cat"] = df_map.soco_path.map(lambda p: Path(p).relative_to(SOCO / "Images").parts[0])
display(pd.crosstab(df_map.soco_cat, df_map.split).head(20))

split,test,train
soco_cat,,
aeroplane,0,4
american_black_bear,19,2
arctic_fox,0,4
ax,0,11
bed,0,4
bench,0,4
bighorn,4,5
boston_bull,3,0
brown_bear,3,16


In [8]:
# 1) la submission copre tutte le query del test?
t = pd.read_csv(SUB / "test.csv")
s = pd.read_csv(SUB / "sample_submission.csv")
print(len(t), len(s), "stessi id:", set(t.id) == set(s.id))

# 2) struttura degli split originali di SOCO
TS = SOCO / "PairAnnotations" / "trainsplits"
for p in sorted(TS.rglob("*"))[:30]:
    print(p.relative_to(SOCO), p.stat().st_size if p.is_file() else "")

577 577 stessi id: True
PairAnnotations/trainsplits/test 
PairAnnotations/trainsplits/test/aeroplane 
PairAnnotations/trainsplits/test/aeroplane/00001-aeroplane_2-aeroplane_40.json 3860
PairAnnotations/trainsplits/test/aeroplane/00002-aeroplane_25-aeroplane_9.json 5419
PairAnnotations/trainsplits/test/aeroplane/00003-aeroplane_11-aeroplane_34.json 5461
PairAnnotations/trainsplits/test/aeroplane/00004-aeroplane_10-aeroplane_32.json 4627
PairAnnotations/trainsplits/test/aeroplane/00005-aeroplane_26-aeroplane_28.json 6492
PairAnnotations/trainsplits/test/aeroplane/00006-aeroplane_13-aeroplane_35.json 4338
PairAnnotations/trainsplits/test/aeroplane/00007-aeroplane_30-aeroplane_31.json 5444
PairAnnotations/trainsplits/test/aeroplane/00008-aeroplane_1-aeroplane_33.json 6521
PairAnnotations/trainsplits/test/aeroplane/00009-aeroplane_37-aeroplane_4.json 5111
PairAnnotations/trainsplits/test/aeroplane/00010-aeroplane_27-aeroplane_29.json 5138
PairAnnotations/trainsplits/test/aeroplane/00011-aer

In [9]:
import re, json

# --- coppie del test (e train) originale di SOCO, dai nomi dei file ---
PAT = re.compile(r"^\d+-(.+)-(.+)\.json$")
def soco_pairs(split):
    out = []
    for f in tqdm(list((SOCO / "PairAnnotations" / "trainsplits" / split).rglob("*.json")), desc=f"pairs {split}"):
        m = PAT.match(f.name)
        a, b = m.groups()
        pa = SOCO / "Images" / a.rsplit("_", 1)[0] / f"{a}.JPEG"
        pb = SOCO / "Images" / b.rsplit("_", 1)[0] / f"{b}.JPEG"
        out.append((str(pa), str(pb)))
    return out

soco_test_pairs  = soco_pairs("test")
soco_train_pairs = soco_pairs("train")
soco_test_imgs   = {p for pr in soco_test_pairs for p in pr}
soco_train_imgs  = {p for pr in soco_train_pairs for p in pr}
print(f"SOCO originale: {len(soco_train_pairs)} coppie train, {len(soco_test_pairs)} coppie test; "
      f"{len(soco_train_imgs)} / {len(soco_test_imgs)} immagini")

# --- mappa sub-path (relativo) -> path SOCO ---
sub2soco = {str(Path(r.sub_path).relative_to(SUB)): r.soco_path for r in df_map.itertuples()}

sub_test_imgs  = {sub2soco[k] for k in sub2soco if k.startswith("images/test")}
sub_train_imgs = {sub2soco[k] for k in sub2soco if k.startswith("images/train")}

# --- coppie del tuo subset, espresse in path SOCO ---
def sub_pairs(js):
    d = json.load(open(SUB / js))
    return {(sub2soco[p["source_image"]], sub2soco[p["target_image"]]) for p in d["pairs"]}
sub_test_pairs  = sub_pairs("test_pairs.json")
sub_train_pairs = sub_pairs("train_pairs.json")

# --- controlli ---
print("coppie sub-test presenti nel test SOCO :", len(sub_test_pairs & set(soco_test_pairs)), "/", len(sub_test_pairs))
print("coppie sub-train presenti nel train SOCO:", len(sub_train_pairs & set(soco_train_pairs)), "/", len(sub_train_pairs))
print("immagini sub-test nel test SOCO         :", len(sub_test_imgs & soco_test_imgs), "/", len(sub_test_imgs))
print("immagini sub-train nel test SOCO (leak?):", len(sub_train_imgs & soco_test_imgs))
print("immagini sub-train nel train SOCO       :", len(sub_train_imgs & soco_train_imgs), "/", len(sub_train_imgs))

# --- quanto test originale NON è nel tuo subset ---
print("coppie test SOCO fuori dal tuo subset   :", len(set(soco_test_pairs) - sub_test_pairs))
print("immagini test SOCO fuori dal tuo subset :", len(soco_test_imgs - sub_test_imgs))

pairs test:   0%|          | 0/10000 [00:00<?, ?it/s]

pairs train:   0%|          | 0/10000 [00:00<?, ?it/s]

SOCO originale: 10000 coppie train, 10000 coppie test; 2000 / 1998 immagini
coppie sub-test presenti nel test SOCO : 8 / 98
coppie sub-train presenti nel train SOCO: 33 / 392
immagini sub-test nel test SOCO         : 67 / 129
immagini sub-train nel test SOCO (leak?): 257
immagini sub-train nel train SOCO       : 260 / 517
coppie test SOCO fuori dal tuo subset   : 9992
immagini test SOCO fuori dal tuo subset : 1931


In [10]:
# ordine invertito?
unordered = lambda S: {frozenset(p) for p in S}
print("sub-test, ignorando la direzione:", len(unordered(sub_test_pairs) & unordered(soco_test_pairs)), "/", len(sub_test_pairs))

# da dove vengono le coppie? guarda cross/intra
for d in ("cross", "intra"):
    base = SOCO / "PairAnnotations" / d
    files = list(base.rglob("*.json"))
    print(d, len(files), [f.relative_to(base).as_posix() for f in files[:2]])

sub-test, ignorando la direzione: 12 / 98
cross 20000 ['jinrikisha/00144-jinrikisha_18-jinrikisha_23.json', 'jinrikisha/00095-cart_25-jinrikisha_13.json']
intra 20000 ['jinrikisha/00033-jinrikisha_11-jinrikisha_1.json', 'jinrikisha/00151-jinrikisha_9-jinrikisha_16.json']


In [11]:
def pairs_from(folder):
    out = {}
    for f in tqdm(list((SOCO / "PairAnnotations" / folder).rglob("*.json")), desc=folder):
        a, b = PAT.match(f.name).groups()
        pa = str(SOCO / "Images" / a.rsplit("_", 1)[0] / f"{a}.JPEG")
        pb = str(SOCO / "Images" / b.rsplit("_", 1)[0] / f"{b}.JPEG")
        out[(pa, pb)] = f
    return out

intra, cross = pairs_from("intra"), pairs_from("cross")
print("intra ⊇ trainsplits?", len((set(soco_train_pairs) | set(soco_test_pairs)) & set(intra)), "/ 20000")

U = lambda S: {frozenset(p) for p in S}
for name, sub in [("sub-test", sub_test_pairs), ("sub-train", sub_train_pairs)]:
    in_intra = len(U(sub) & U(intra)); in_cross = len(U(sub) & U(cross))
    in_any   = len(U(sub) & (U(intra) | U(cross)))
    print(f"{name}: {len(sub)} coppie | intra {in_intra} | cross {in_cross} | almeno una {in_any}")

# categoria sorgente vs target delle coppie del tuo test: quante sono cross-category?
cat = lambda p: Path(p).parent.name
print("coppie sub-test con categorie diverse:", sum(cat(a) != cat(b) for a, b in sub_test_pairs), "/", len(sub_test_pairs))

intra:   0%|          | 0/20000 [00:00<?, ?it/s]

cross:   0%|          | 0/20000 [00:00<?, ?it/s]

intra ⊇ trainsplits? 3336 / 20000
sub-test: 98 coppie | intra 98 | cross 2 | almeno una 98
sub-train: 392 coppie | intra 392 | cross 35 | almeno una 392
coppie sub-test con categorie diverse: 0 / 98
